# Final Feature Engineering and Frozen Credit-Scoring Model

## Purpose

This notebook documents the final experimental stage of the credit-scoring capstone project.

Following the initial benchmarking, stacking, calibration, financial-analysis, explainability, and deployment stages, additional feature-engineering experiments were conducted in response to supervisor feedback.

The purpose of this stage was to determine whether financially meaningful engineered features improved the three modelling families:

- Logistic Regression
- XGBoost
- PyTorch Multi-Layer Perceptron (MLP)

The experiments considered:

1. employment normalisation;
2. log transformations of skewed financial variables;
3. financial ratio features;
4. model-specific feature configurations;
5. feature-engineered out-of-fold stacking; and
6. probability calibration.

The final selected pipeline was then evaluated on the reserved held-out test set.

> **Important:** The final test set is treated as a locked evaluation set. Results from the final test evaluation are reported but are not used for additional model tuning or feature selection.

## 1. Experimental Design

The Home Credit `application_train.csv` dataset contains 307,511 applications.

The original stratified split was:

| Partition | Observations | Approximate Share | Purpose |
|---|---:|---:|---|
| Training | 215,257 | 70% | Initial model development |
| Validation | 46,127 | 15% | Feature/model comparison |
| Test | 46,127 | 15% | Locked final evaluation |

Feature-engineering decisions were made using development-stage evidence.

After the final feature configurations were selected, the original training and validation partitions were combined into an approximately 85% development dataset.

Five-fold out-of-fold (OOF) predictions were generated within the development data for the final stacking procedure.

The original 15% test partition remained reserved until the modelling and calibration strategy had been selected.

## 2. Core Data Preparation

The core cleaned dataset uses 121 predictors:

- 105 numerical variables
- 16 categorical variables

The Home Credit `DAYS_EMPLOYED = 365243` sentinel value is treated as an anomaly.

The core engineered variables include:

- `AGE_YEARS`
- `EMPLOYED_YEARS`
- `DAYS_EMPLOYED_ANOM`

The following are excluded from the final predictor set where appropriate:

- `SK_ID_CURR`
- `TARGET`
- original `DAYS_BIRTH`
- original `DAYS_EMPLOYED`

This notebook focuses on the later feature-engineering experiments rather than repeating the full exploratory analysis documented in `01_Data_EDA.ipynb`.

## 3. Employment-Normalisation Experiment

The first additional experiment tested whether employment duration was more informative when expressed relative to an applicant's approximate potential working life.

### Engineered Feature

The employment ratio was defined as:

**EMPLOYMENT_RATIO = EMPLOYED_YEARS / (AGE_YEARS - 18)**

The ratio was created only for valid employment observations. The existing employment-anomaly treatment was retained.

### Validation Results

| Configuration | ROC-AUC | PR-AUC | Brier Score | F1 |
|---|---:|---:|---:|---:|
| Baseline | 0.75736 | 0.24901 | 0.067659 | 0.04099 |
| Employment Ratio | 0.75709 | 0.24760 | 0.067706 | 0.03839 |

### Decision

**Rejected.**

The employment ratio did not improve the principal validation metrics.

This experiment demonstrates that a financially or conceptually meaningful engineered variable does not necessarily provide additional predictive information.

## 4. Log-Transformation Experiment

Several important financial variables were strongly right-skewed. Log transformations were tested to determine whether reducing this skewness improved predictive performance.

### Variables Evaluated

The experiment considered log transformations of:

- Income
- Credit amount
- Annuity
- Goods price

### Effect on Skewness

| Variable | Original Skewness | Log-Transformed Skewness |
|---|---:|---:|
| Income | 391.56 | 0.17 |
| Credit | 1.23 | -0.34 |
| Annuity | 1.58 | -0.35 |
| Goods Price | 1.35 | -0.29 |

The transformations substantially reduced the skewness of the financial variables, particularly income.

### Logistic Regression Results

| Configuration | ROC-AUC | PR-AUC | Brier Score | F1 | Mean PD |
|---|---:|---:|---:|---:|---:|
| Baseline | 0.748083 | 0.230662 | 0.068462 | 0.027282 | 0.080843 |
| Log Income | 0.747991 | 0.230622 | 0.068464 | 0.026247 | 0.080834 |
| All Logs | 0.749224 | 0.232986 | 0.068357 | 0.030374 | 0.080810 |

For Logistic Regression, applying all evaluated log transformations produced a small improvement over the baseline.

### XGBoost Results

| Configuration | ROC-AUC | PR-AUC | Brier Score | F1 | Mean PD |
|---|---:|---:|---:|---:|---:|
| Baseline | 0.757062 | 0.247693 | 0.067716 | 0.037843 | 0.080319 |
| Log Income | 0.757292 | 0.247979 | 0.067694 | 0.040890 | 0.080485 |
| All Logs | 0.756453 | 0.247515 | 0.067729 | 0.040467 | 0.080409 |

For XGBoost, the improvements from log transformations were small and inconsistent. Log income produced a slight improvement, while applying all log transformations slightly reduced ROC-AUC.

### Decision

**Partially retained.**

Log transformations were useful for some model families but were not universally beneficial.

The experiment supported using log-transformed financial variables as part of the final Logistic Regression and MLP configurations, while XGBoost was not given the complete set of log transformations.

This supports the use of model-specific feature engineering rather than forcing identical engineered features into every model.

## 5. Financial-Ratio Experiment

The next experiment created financial ratios that represent relationships between an applicant's income, requested credit, annuity payments, and goods price.

The purpose was to provide information about financial burden and loan structure that may not be captured as clearly when the monetary variables are considered independently.

### Engineered Ratios

The following ratios were evaluated:

- **CREDIT_INCOME_RATIO = AMT_CREDIT / AMT_INCOME_TOTAL**
- **ANNUITY_INCOME_RATIO = AMT_ANNUITY / AMT_INCOME_TOTAL**
- **CREDIT_GOODS_RATIO = AMT_CREDIT / AMT_GOODS_PRICE**
- **ANNUITY_CREDIT_RATIO = AMT_ANNUITY / AMT_CREDIT**

### Logistic Regression Results

| Configuration | ROC-AUC | PR-AUC | Brier Score | F1 | Mean PD |
|---|---:|---:|---:|---:|---:|
| Baseline | 0.748083 | 0.230662 | 0.068462 | 0.027282 | 0.080843 |
| All Ratios | 0.749519 | 0.232827 | 0.068357 | 0.029820 | 0.080845 |
| Ratios + All Logs | 0.750621 | 0.235055 | 0.068263 | 0.031307 | 0.080805 |

The financial ratios improved Logistic Regression over the baseline.

Combining the ratios with the log-transformed financial variables produced the strongest Logistic Regression configuration in this experiment.

### XGBoost Results

| Configuration | ROC-AUC | PR-AUC | Brier Score | F1 | Mean PD |
|---|---:|---:|---:|---:|---:|
| Baseline | 0.757062 | 0.247693 | 0.067716 | 0.037843 | 0.080319 |
| All Ratios | 0.765558 | 0.255163 | 0.067298 | 0.048831 | 0.080204 |
| Ratios + Log Income | 0.765521 | 0.255914 | 0.067273 | 0.051755 | 0.080260 |

The ratio features produced a clear improvement for XGBoost.

Compared with the baseline, the all-ratios configuration increased ROC-AUC from 0.757062 to 0.765558 and reduced the Brier Score from 0.067716 to 0.067298.

Among the individual ratio experiments, **ANNUITY_CREDIT_RATIO** provided the strongest observed XGBoost result.

### Decision

**Financial ratios retained.**

The final model-specific decisions were:

- Logistic Regression: retain ratios together with the evaluated log transformations.
- XGBoost: retain the financial-ratio feature set.

Although adding log income to the XGBoost ratio configuration slightly improved PR-AUC, Brier Score, and F1, its ROC-AUC was slightly lower than the all-ratios configuration.

The final controlled XGBoost configuration therefore retained the financial ratios without adding the full log-transformation set.

### Key Finding

The ratio experiment provided stronger evidence of predictive improvement than the employment-normalisation experiment or log transformations alone.

It also demonstrated that feature engineering affected the modelling families differently, supporting the use of model-specific feature configurations.

## 6. MLP Feature-Engineering Experiment

The feature-engineering comparison was also performed for the PyTorch Multi-Layer Perceptron.

To make the comparison controlled, the same reconstructed MLP architecture and training configuration were used across the feature sets.

### Fixed MLP Architecture

The architecture used for this experiment was:

Input → 128 → ReLU → Dropout (0.20) → 64 → ReLU → Dropout (0.20) → Output

Training configuration:

- Loss function: BCEWithLogitsLoss
- Optimizer: Adam
- Learning rate: 0.001
- Batch size: 2048
- Epochs: 12

Keeping the architecture fixed allowed the experiment to focus on the effect of feature engineering rather than simultaneously changing the neural-network structure.

### Results

| Configuration | ROC-AUC | PR-AUC | Brier Score | F1 | Mean PD |
|---|---:|---:|---:|---:|---:|
| Baseline | 0.748796 | 0.230121 | 0.068447 | 0.001609 | 0.081875 |
| Ratios | 0.751293 | 0.237145 | 0.068231 | 0.010125 | 0.081184 |
| Ratios + Logs | 0.752253 | 0.240995 | 0.068155 | 0.001610 | 0.077509 |

Both engineered configurations improved ROC-AUC, PR-AUC, and Brier Score compared with the baseline MLP.

The ratios + logs configuration achieved the strongest ROC-AUC, PR-AUC, and Brier Score of the three evaluated configurations.

The ratios-only configuration produced the highest F1 at the fixed 0.5 threshold.

### Decision

**Ratios + logs retained.**

Because the project primarily uses the model output as a probability rather than only as a binary classification, the combination of ROC-AUC, PR-AUC, and Brier Score was given greater importance than F1 at one fixed threshold.

The final MLP therefore uses financial ratios together with the evaluated log-transformed financial features.

### MLP Reconstruction Limitation

The exact configuration of the earliest historical MLP experiment was not available for complete reconstruction.

Therefore, this experiment should be interpreted as a controlled feature-engineering comparison using a fixed reconstructed MLP architecture.

It should not be interpreted as an exact reproduction of every earlier MLP result.

## 7. Final Feature Selection and Stacking Experiment

Based on the previous feature-engineering experiments, different feature configurations were retained for each modelling family.

### Final Base-Model Configurations

| Model | Selected Feature Configuration |
|---|---|
| Logistic Regression | Financial Ratios + Log Transformations |
| XGBoost | Financial Ratios |
| MLP | Financial Ratios + Log Transformations |

The same engineered feature set was not forced onto all three models because the experiments showed that the modelling families responded differently to the transformations.

### Feature-Engineered Ensemble

The selected base models were combined using five-fold out-of-fold stacking:

LR (Ratios + Logs)  
+ XGBoost (Ratios)  
+ MLP (Ratios + Logs)  
→ 5-Fold OOF Probabilities  
→ Meta Logistic Regression  
→ Raw Ensemble PD

### Development-Stage Stack Comparison

| Configuration | ROC-AUC | PR-AUC | Brier Score | F1 | Mean PD |
|---|---:|---:|---:|---:|---:|
| Baseline Stack | 0.754343 | 0.241684 | 0.068609 | 0.100928 | 0.081318 |
| Feature-Engineered Stack | 0.762650 | 0.248052 | 0.068323 | 0.106317 | 0.079581 |

### Change from Baseline Stack

The feature-engineered stack produced approximately:

- ROC-AUC improvement: +0.008306
- PR-AUC improvement: +0.006368
- Brier Score improvement: -0.000286
- F1 improvement: +0.005389

A lower Brier Score represents better probability accuracy.

### Decision

**Feature-engineered stack retained.**

The feature-engineered ensemble improved all four reported development-stage metrics compared with the baseline stack.

This provided the evidence for replacing the earlier baseline ensemble with the model-specific feature-engineered ensemble.

### Final Ensemble Components

The selected ensemble therefore consists of:

1. Logistic Regression using ratios + logs
2. XGBoost using ratios
3. MLP using ratios + logs
4. Five-fold out-of-fold probability generation
5. Logistic Regression meta-learner

Probability calibration is evaluated separately in the next stage.

## 8. Development-Stage Calibration Comparison

After selecting the feature-engineered stack, three probability outputs were compared during development:

- Raw stacking probability
- Platt scaling
- Isotonic regression

The purpose of calibration was to improve the quality of the predicted probabilities because the final output is interpreted as Probability of Default (PD) and is subsequently used in financial calculations.

### Baseline Stack Calibration

| Output | ROC-AUC | PR-AUC | Brier Score |
|---|---:|---:|---:|
| Raw | 0.761572 | 0.249195 | 0.068227 |
| Platt | 0.761572 | 0.249195 | 0.069552 |
| Isotonic | 0.760955 | 0.239098 | 0.067603 |

For the baseline stack, isotonic regression produced the lowest Brier Score.

### Feature-Engineered Stack Calibration

| Output | ROC-AUC | PR-AUC | Brier Score |
|---|---:|---:|---:|
| Raw | 0.768947 | 0.255870 | 0.067943 |
| Platt | 0.768947 | 0.255870 | 0.069377 |
| Isotonic | 0.768072 | 0.246190 | 0.067118 |

The raw feature-engineered stack produced the highest ROC-AUC and PR-AUC.

However, isotonic regression produced the lowest Brier Score:

**Raw Brier Score: 0.067943**

**Isotonic Brier Score: 0.067118**

### Interpretation

Calibration involves a trade-off between discrimination and probability quality.

ROC-AUC and PR-AUC primarily measure the model's ability to rank higher-risk applicants above lower-risk applicants.

Brier Score measures the accuracy of the predicted probabilities.

Because this project uses the final probability as PD in Expected Loss and financial scenario calculations, probability quality was considered particularly important.

### Decision

**Isotonic regression selected.**

Isotonic calibration was selected based on the development-stage comparison because it achieved the lowest Brier Score.

Importantly, this decision was made before the reserved final test set was opened.

The calibration strategy was then frozen as part of the final modelling pipeline.

The final test set was therefore used for evaluation rather than for choosing between raw, Platt, and isotonic probabilities.

## 9. Final Frozen Pipeline and Locked Test Evaluation

After completing the feature-engineering, stacking, and calibration experiments, the final modelling pipeline was frozen.

### Final Pipeline

The selected pipeline was:

**Logistic Regression (Ratios + Logs)**  
+ **XGBoost (Ratios)**  
+ **MLP (Ratios + Logs)**  
→ **5-Fold OOF Predictions**  
→ **Meta Logistic Regression**  
→ **Isotonic Calibration**  
→ **Final PD**

For final model development, the original training and validation partitions were combined into an approximately 85% development dataset.

Five-fold out-of-fold predictions were generated within the development data.

The original 15% test partition remained reserved until the feature configurations, stacking approach, and calibration strategy had been selected.

### Locked Test Set

The final held-out test set contained:

**46,127 observations**

The default prevalence was approximately:

**8.07%**

### Final Test Results

| Model / Output | ROC-AUC | PR-AUC | Brier Score | F1 @ 0.5 | Mean PD |
|---|---:|---:|---:|---:|---:|
| LR - Ratios + Logs | 0.752869 | 0.228018 | 0.068404 | 0.026646 | 0.080790 |
| XGBoost - Ratios | 0.768588 | 0.256410 | 0.067168 | 0.051374 | 0.080327 |
| MLP - Ratios + Logs | 0.754493 | 0.231109 | 0.068268 | 0.015835 | 0.083818 |
| Raw Stack | 0.768823 | 0.254761 | 0.068026 | 0.118064 | 0.081715 |
| Final Isotonic PD | 0.768251 | 0.245151 | 0.067306 | 0.025729 | 0.080942 |

### Interpretation

The strongest individual model was XGBoost with financial-ratio features.

XGBoost achieved:

- ROC-AUC = 0.768588
- PR-AUC = 0.256410
- Brier Score = 0.067168

The raw stacking ensemble achieved the highest overall ROC-AUC:

**ROC-AUC = 0.768823**

The final isotonic output achieved:

- ROC-AUC = 0.768251
- PR-AUC = 0.245151
- Brier Score = 0.067306

Although XGBoost achieved a slightly lower Brier Score than the final isotonic output on the locked test set, the pipeline was not changed after observing this result.

Isotonic calibration had already been selected during development and was therefore retained as the frozen final PD mapping.

### Why the Model Was Not Changed After the Test

The purpose of the final test set is to estimate how the already-selected modelling pipeline performs on unseen data.

Using the final test results to switch back to XGBoost, remove calibration, choose another calibrator, or tune the classification threshold would turn the test set into another model-selection dataset.

Therefore, no additional model selection or tuning was performed after these results were observed.

## 10. Interpretation of F1 at the 0.5 Threshold

The final isotonic probability has a substantially lower F1 score at the fixed classification threshold of 0.5.

This result should be interpreted carefully.

F1 is a threshold-dependent classification metric. It depends on converting probabilities into binary predictions using a selected cutoff.

In this experiment, the reported F1 uses:

**Threshold = 0.5**

The primary objective of the final system, however, is to estimate a continuous Probability of Default rather than simply classify applicants as default or non-default.

Isotonic calibration changes the probability mapping. As a result, the number of applicants whose predicted probability exceeds 0.5 can change substantially even when applicant ranking remains relatively similar.

This can be seen from the discrimination results:

- Raw Stack ROC-AUC = 0.768823
- Final Isotonic ROC-AUC = 0.768251

The ranking performance changed only slightly, while F1 at the fixed 0.5 threshold changed considerably.

Therefore, F1 is interpreted together with:

- ROC-AUC
- PR-AUC
- Brier Score
- Mean PD
- the intended probability-based use of the model

No alternative threshold was selected using the locked final test set.

## 11. Meta-Learner

The stacking ensemble uses three probabilities as its meta-features:

- Logistic Regression PD
- XGBoost PD
- MLP PD

These probabilities are combined using a Logistic Regression meta-learner.

### Final Meta-Model Coefficients

| Base-Model Probability | Coefficient |
|---|---:|
| Logistic Regression | 0.3622 |
| XGBoost | 5.9710 |
| MLP | 2.4481 |

XGBoost received the largest coefficient in the final meta-model, followed by the MLP and Logistic Regression.

This is consistent with XGBoost being the strongest individual model in the final experiment.

However, these coefficients should not be interpreted as feature importance for the original applicant variables.

They describe how the meta-logistic model combines the three base-model probabilities.

For example:

**XGBoost coefficient = 5.9710**

does not mean that XGBoost is "5.971 times more important" than another model.

The coefficient operates within the Logistic Regression meta-model and depends on the scale and relationship of the three probability inputs.

## 12. Calibration Implementation Limitation

The final calibration procedure contains an implementation limitation that should be documented explicitly.

During development, the OOF meta-data was divided into a meta-training portion and a calibration portion.

A calibration-stage Logistic Regression meta-model was fitted using the meta-training portion.

This model generated raw ensemble probabilities for the calibration portion, and isotonic regression was fitted using those probabilities.

Conceptually:

**OOF Meta-Data**  
→ **Meta-Training / Calibration Split**  
→ **Calibration-Stage MetaLR**  
→ **Calibration Raw Probabilities**  
→ **Isotonic Calibrator**

For final inference, however, the final Logistic Regression meta-model was fitted using the complete development OOF meta-data.

The final prediction path therefore uses:

**Base-Model Probabilities**  
→ **Full-Development MetaLR**  
→ **Raw Final Probability**  
→ **Previously Fitted Isotonic Calibrator**

### Limitation

The isotonic calibrator was fitted to probabilities generated by the calibration-stage meta-model, but it is applied during final inference to probabilities generated by the full-development meta-model.

The two meta-models:

- use the same Logistic Regression modelling approach;
- use the same three base-model probability inputs;
- belong to the same development procedure;

but they are not the identical fitted model.

Therefore, the probability distribution used to train the isotonic mapping is not guaranteed to be exactly identical to the probability distribution produced by the final full-development meta-model.

### Why This Was Not Changed After Final Testing

This limitation was identified after the final evaluation procedure had already been completed.

Changing the calibration procedure and then comparing the revised model on the same final test set would introduce another round of test-driven model selection.

The current final results are therefore retained and the limitation is reported transparently.

### Future Improvement

A future implementation could use a cleaner calibration design, such as:

- cross-fitted calibration;
- nested validation;
- a dedicated calibration partition;
- or another procedure that ensures the calibrator is fitted to probabilities generated consistently with the final inference mapping.

This is treated as a methodological limitation rather than evidence that the reported final experiment should be retrospectively changed.

## 13. Final Model Artifacts

The frozen feature-engineered pipeline was exported for use by the Flask prediction service.

The final artifacts are stored in:

**models_final/**

### Logistic Regression

- `lr_preprocessor.joblib`
- `lr_model.joblib`

### XGBoost

- `xgb_preprocessor.joblib`
- `xgb_model.joblib`

### MLP

- `mlp_preprocessor.joblib`
- `mlp_state.pt`
- `mlp_metadata.json`

### Ensemble and Calibration

- `meta_logistic.joblib`
- `isotonic_calibrator.joblib`

### Feature Configuration

- `feature_config.json`

The separate preprocessors are required because the three base models use different final feature configurations.

The deployed prediction sequence is:

Applicant Data  
→ Model-Specific Feature Engineering  
→ LR / XGBoost / MLP Probabilities  
→ Meta Logistic Regression  
→ Isotonic Calibration  
→ Final Probability of Default

The final model artifacts are used by `deployment/model_service.py`.

## 14. Final Experimental Conclusion and Model Freeze

The feature-engineering stage showed that engineered variables did not affect all modelling families equally.

The employment-normalisation feature was rejected because it did not improve validation performance.

Log transformations substantially reduced skewness but produced only modest and model-dependent predictive improvements.

Financial ratios provided the clearest improvement, particularly for XGBoost.

Based on the controlled experiments, the final base-model configurations were:

- Logistic Regression: ratios + logs
- XGBoost: ratios
- MLP: ratios + logs

Combining these configurations through five-fold out-of-fold stacking improved the development-stage ensemble compared with the earlier baseline stack.

Isotonic regression was selected during development based on probability calibration performance before the final test set was evaluated.

The final frozen pipeline is:

**LR (Ratios + Logs) + XGBoost (Ratios) + MLP (Ratios + Logs) → 5-Fold OOF → MetaLR → Isotonic → Final PD**

The final held-out test evaluation produced:

- ROC-AUC: 0.768251
- PR-AUC: 0.245151
- Brier Score: 0.067306
- Mean PD: 0.080942

The model is now considered frozen for the current Capstone I project.

No further feature selection, calibration selection, threshold optimisation, or hyperparameter tuning should be performed using the final test results.

Further modelling changes should only be made if:

1. a genuine implementation error is identified; or
2. new modelling work is specifically required following supervisor feedback.

Otherwise, the remaining project work should focus on documentation, reproducibility, deployment evidence, report preparation, and presentation.